In [1]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
import torch



In [4]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,   # 双重量化，再省一点显存
    bnb_4bit_quant_type="nf4",        # NF4 格式，精度更好
    llm_int8_enable_fp32_cpu_offload=True,  # 允许部分层在 CPU 以 fp32 运行
)

model_id = "google/gemma-4-26b-a4b-it"

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    max_memory={0: "14GiB", "cpu": "28GiB"},  # 显式分配，给系统留点余量
)

RuntimeError: unable to mmap 49907246508 bytes from file </home/ubuntu/.cache/huggingface/hub/models--google--gemma-4-26b-a4b-it/snapshots/20da991ab4afab98e8f910c4a2e8f4fbefc404ad/model-00001-of-00002.safetensors>: Cannot allocate memory (12)

In [6]:
from transformers import AutoConfig, AutoModelForImageTextToText
import torch

with torch.device("meta"):
    model = AutoModelForImageTextToText.from_config(
        AutoConfig.from_pretrained("google/gemma-4-26b-a4b-it")
    )

print(model)

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (vision_tower): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=1152, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-26): 27 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=1152, out_features=1152, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=1152, out_features=1152, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=1152, out_features=1152, bias=False)
              )
              (o_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=1152, out_features=1152, 